# Notebook 04: Semantic Ingredient Labeling

**Phase:** Semantic Classification (Thesis Plan Deliverable 1)

**Purpose:** Annotate individual ingredients with semantic category labels for training a semantic classification model.

## Overview

This notebook assigns **semantic category labels** to each ingredient from Notebook 03.
Labels follow the taxonomy in `configs/semantic_categories.json` and include:
- **Functional categories**: food_additive, preservative, emulsifier, thickener, etc.
- **Source categories**: animal_derived, plant_derived, milk_derivative, etc.
- **Nutrient categories**: fat_source, protein_source, carbohydrate_source, etc.

## Workflow

1. Load parsed ingredients from Notebook 03
2. Apply rule-based labeling using `ingredient_to_categories()` from `semantic_utils`
3. Review and correct labels for ambiguous ingredients
4. Build the multi-label training dataset
5. Save labeled dataset
6. Export annotation statistics for the next annotation cycle

In [1]:
import os
import pandas as pd
import numpy as np
from collections import Counter, defaultdict

from utils.data_utils import get_data_directories, save_metadata
from utils.semantic_utils import (
    ingredient_to_categories,
    get_semantic_category_list,
    semantic_labels_to_binary,
    binary_to_semantic_labels,
    get_category_groups,
)

print("Imports successful.")
print(f"Semantic categories: {len(get_semantic_category_list())}")

Imports successful.
Semantic categories: 43


## 1. Load Parsed Ingredients

In [2]:
dirs = get_data_directories()

# Load parsed ingredients from Notebook 03
ingredient_path = os.path.join(dirs['interim'], 'parsed_ingredients.csv')
ingredient_df = pd.read_csv(ingredient_path)
print(f"Loaded {len(ingredient_df)} ingredient instances")
print(f"Unique ingredient terms: {ingredient_df['ingredient'].nunique()}")
print(f"Source products: {ingredient_df['product_code'].nunique()}")

Loaded 18623 ingredient instances
Unique ingredient terms: 4664
Source products: 1057


## 2. Apply Rule-Based Semantic Labeling

The `ingredient_to_categories()` function maps known ingredients to their semantic categories.

In [3]:
# Apply rule-based labeling to each unique ingredient
unique_ingredients = ingredient_df['ingredient'].unique()
print(f"Applying rule-based labels to {len(unique_ingredients)} unique ingredients...")

label_map = {}
for ing in unique_ingredients:
    cats = ingredient_to_categories(ing)
    label_map[ing] = cats

# Count how many were matched vs. unknown
matched = sum(1 for v in label_map.values() if v)
unknown = sum(1 for v in label_map.values() if not v)
print(f"  Matched: {matched} ingredients ({matched/len(unique_ingredients):.1%})")
print(f"  Unknown: {unknown} ingredients ({unknown/len(unique_ingredients):.1%})")

# Display coverage by instance count
total_instances = len(ingredient_df)
matched_instances = sum(c for ing, c in Counter(ingredient_df['ingredient']).items() if label_map.get(ing))
print(f"  Instance coverage: {matched_instances}/{total_instances} ({matched_instances/total_instances:.1%})")

Applying rule-based labels to 4664 unique ingredients...
  Matched: 1997 ingredients (42.8%)
  Unknown: 2667 ingredients (57.2%)
  Instance coverage: 10588/18623 (56.9%)


## 3. Review Unknown Ingredients

Ingredients without automatic labels need manual annotation to expand the rule base.

In [4]:
# Find unknown ingredients (most frequent first)
ingredient_counts = Counter(ingredient_df['ingredient'])
unknown_ingredients = [
    (ing, count, label_map.get(ing, []))
    for ing, count in ingredient_counts.most_common()
    if not label_map.get(ing)
]

print(f"Unknown ingredients needing annotation: {len(unknown_ingredients)}")
print(f"\nTop 30 unknown ingredients by frequency:")
print(f"{'Ingredient':30s} {'Count':6s}")
print("-" * 36)
for ing, count, _ in unknown_ingredients[:30]:
    print(f"{ing:30s} {count:6d}")

# Export for manual annotation
unknown_df = pd.DataFrame(unknown_ingredients, columns=['ingredient', 'count', 'labels'])
unknown_path = os.path.join(dirs['interim'], 'unknown_ingredients_for_labeling.csv')
unknown_df.to_csv(unknown_path, index=False)
print(f"\nSaved {len(unknown_df)} unknown ingredients to: {unknown_path}")

Unknown ingredients needing annotation: 2667

Top 30 unknown ingredients by frequency:
Ingredient                     Count 
------------------------------------
spices                            193
emulsifier                        171
stabilizer                        127
antioxidant                       112
acidity regulator                  99
preservative                       97
flavor enhancers                   93
flavor enhancer                    93
silicon dioxide                    91
emulsifiers                        89
sodium citrate                     79
acidity regulators                 76
sucralose                          75
sweetener                          67
tartrazine                         65
mono -                             64
stabilizers                        58
sunset yellow                      55
vegetable fat                      52
palm olein                         52
vitamins                           50
tbhq                               50
an

## 4. Build Multi-Label Training Dataset

Create the binary matrix for semantic classification training.

In [5]:
categories = get_semantic_category_list()

# Map each ingredient to its binary label vector
ingredient_df['semantic_labels'] = ingredient_df['ingredient'].map(
    lambda x: label_map.get(x, [])
)
ingredient_df['label_vector'] = ingredient_df['semantic_labels'].apply(
    lambda x: semantic_labels_to_binary(x, categories)
)

print(f"Training data shape: {len(ingredient_df)} x {len(categories)} labels")

# Quick stats
label_matrix = np.array(ingredient_df['label_vector'].tolist())
print(f"Positive labels per ingredient: {label_matrix.sum(axis=1).mean():.2f} (mean)")
print(f"\nLabel distribution:")
for i, cat in enumerate(categories):
    count = label_matrix[:, i].sum()
    pct = count / len(ingredient_df)
    print(f"  {cat:30s} {int(count):5d} ({pct:.1%})")

Training data shape: 18623 x 43 labels
Positive labels per ingredient: 1.63 (mean)

Label distribution:
  food_additive                   2526 (13.6%)
  preservative                     450 (2.4%)
  flavor_enhancer                  507 (2.7%)
  sweetener                       1474 (7.9%)
  emulsifier                       349 (1.9%)
  stabilizer                       361 (1.9%)
  thickener                        611 (3.3%)
  antioxidant                      477 (2.6%)
  acidulant                        474 (2.5%)
  colorant                         444 (2.4%)
  fat_source                      1588 (8.5%)
  oil_source                      1030 (5.5%)
  protein_source                  1146 (6.2%)
  carbohydrate_source             2303 (12.4%)
  animal_derived                  1490 (8.0%)
  plant_derived                   3118 (16.7%)
  milk_derivative                 1327 (7.1%)
  egg_derivative                   159 (0.9%)
  soy_derivative                   584 (3.1%)
  wheat_derivative 

## 5. Save Labeled Dataset

In [6]:
# Build final labeling dataset (ingredient-level) -- convert lists to tuples
# for hashable deduplication, then restore
label_df = ingredient_df[['ingredient', 'semantic_labels']].copy()
label_df['semantic_labels'] = label_df['semantic_labels'].apply(tuple)
label_df = label_df.drop_duplicates()
label_df['semantic_labels'] = label_df['semantic_labels'].apply(list)
label_path = os.path.join(dirs['final'], 'semantic_labels.csv')
label_df.to_csv(label_path, index=False)
print(f"Saved semantic labels to: {label_path}")

# Also save the full ingredient-level training matrix
training_path = os.path.join(dirs['final'], 'semantic_training_data.csv')
ingredient_df.to_csv(training_path, index=False)
print(f"Saved full training data to: {training_path}")

# Save metadata
n_unique = len(unique_ingredients)
metadata = {
    "notebook": "04_semantic_labeling",
    "total_ingredients": len(ingredient_df),
    "unique_ingredients": n_unique,
    "known_ingredients": matched,
    "unknown_ingredients": unknown,
    "num_categories": len(categories),
    "coverage_rate": round(matched / n_unique, 4) if n_unique else 0,
}
save_metadata(metadata, os.path.join(dirs['final'], 'semantic_labeling_metadata.json'))
print("\nMetadata saved.")

Saved semantic labels to: /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/data/final/semantic_labels.csv
Saved full training data to: /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/data/final/semantic_training_data.csv

Metadata saved.


## 6. Summary

In [7]:
print("=" * 50)
print("SEMANTIC LABELING SUMMARY")
print("=" * 50)
print(f"  Unique ingredients:        {len(unique_ingredients)}")
print(f"  Rule-matched:              {matched} ({matched/len(unique_ingredients):.1%})")
print(f"  Needing manual annotation: {unknown} ({unknown/len(unique_ingredients):.1%})")
print(f"  Semantic categories used:  {len(categories)}")
print(f"  Avg labels per ingredient: {label_matrix.sum(axis=1).mean():.2f}")
print()
print("📋 Next step: Run 05_semantic_model.ipynb to train")

SEMANTIC LABELING SUMMARY
  Unique ingredients:        4664
  Rule-matched:              1997 (42.8%)
  Needing manual annotation: 2667 (57.2%)
  Semantic categories used:  43
  Avg labels per ingredient: 1.63

📋 Next step: Run 05_semantic_model.ipynb to train
